In [1]:
import os
import pandas as pd

RAW_DIR = "../data/raw"
PROC_DIR = "../data/processed"
os.makedirs(PROC_DIR, exist_ok=True)

DARKO_PATH = os.path.join(RAW_DIR, "darko_2025_26.csv")
MINUTES_PATH = os.path.join(RAW_DIR, "minutes_2025_26.csv")

DARKO_PATH, MINUTES_PATH

('../data/raw/darko_2025_26.csv', '../data/raw/minutes_2025_26.csv')

In [2]:
darko = pd.read_csv(DARKO_PATH)
minutes = pd.read_csv(MINUTES_PATH)

print("darko shape:", darko.shape)
print("minutes shape:", minutes.shape)

darko.head()

darko shape: (526, 30)
minutes shape: (546, 33)


,nba_id,Team,Player,Experience,DPM,DPM Improvement,O-DPM,D-DPM,Box DPM,Box O-DPM,...,FTARate%,USG%,REB/100,AST/100,AST%,BLK/100,BLK%,STL/100,STL%,TOV/100
0,203999,Denver Nuggets,Nikola Jokic,Vet,6.71,0.03,5.30,1.41,4.36,4.35,...,0.393,0.311,15.895,14.200,0.456,1.018,0.018,2.192,0.021,5.383
1,202695,Los Angeles Clippers,Kawhi Leonard,Vet,6.20,1.27,4.50,1.70,6.89,5.16,...,0.254,0.329,9.477,5.152,0.191,0.908,0.018,2.661,0.026,3.308
2,1628983,Oklahoma City Thunder,Shai Gilgeous-Alexander,Vet,6.06,-0.19,5.47,0.59,3.64,4.14,...,0.430,0.323,6.408,8.223,0.302,1.164,0.021,2.078,0.021,3.327
3,203507,Milwaukee Bucks,Giannis Antetokounmpo,Vet,6.01,-0.20,4.33,1.68,4.33,3.68,...,0.618,0.327,17.486,8.726,0.320,1.355,0.023,1.534,0.015,5.010
4,1628378,Cleveland Cavaliers,Donovan Mitchell,Vet,5.42,0.73,5.59,-0.17,3.39,4.73,...,0.289,0.314,6.653,8.057,0.272,0.319,0.010,2.107,0.021,4.028


In [3]:
print("darko columns:\n", list(darko.columns))
print("\nminutes columns:\n", list(minutes.columns))

darko columns:
 ['nba_id', 'Team', 'Player', 'Experience', 'DPM', 'DPM Improvement', 'O-DPM', 'D-DPM', 'Box DPM', 'Box O-DPM', 'Box D-DPM', 'FGA/100', 'FG2%', 'FG3A/100', 'FG3%', 'FG3ARate%', 'RimFGA/100', 'RimFG%', 'FTA/100', 'FT%', 'FTARate%', 'USG%', 'REB/100', 'AST/100', 'AST%', 'BLK/100', 'BLK%', 'STL/100', 'STL%', 'TOV/100']

minutes columns:
 ['Rk', 'Player', 'Age', 'Team', 'Pos', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Trp-Dbl', 'Awards', 'Player-additional']


In [4]:
# DARKO -> standard names
darko = darko.rename(columns={
    "Player": "player_name",
    "Team": "team",
    "DPM": "impact_dpm"
})

# Minutes -> standard names
minutes = minutes.rename(columns={
    "Player": "player_name",
    "Team": "team",
    "MP": "minutes"
})

# Strip whitespace to prevent silent merge failures
for col in ["player_name", "team"]:
    darko[col] = darko[col].astype(str).str.strip()
    minutes[col] = minutes[col].astype(str).str.strip()

darko[["player_name","team","impact_dpm"]].head()

,player_name,team,impact_dpm
0,Nikola Jokic,Denver Nuggets,6.71
1,Kawhi Leonard,Los Angeles Clippers,6.20
2,Shai Gilgeous-Alexander,Oklahoma City Thunder,6.06
3,Giannis Antetokounmpo,Milwaukee Bucks,6.01
4,Donovan Mitchell,Cleveland Cavaliers,5.42


In [5]:
merged = darko.merge(
    minutes[["player_name","team","minutes"]],
    on=["player_name","team"],
    how="left"
)

print("merged shape:", merged.shape)
print("missing minutes:", merged["minutes"].isna().sum())

merged.head()

merged shape: (548, 31)
missing minutes: 25


,nba_id,team,player_name,Experience,impact_dpm,DPM Improvement,O-DPM,D-DPM,Box DPM,Box O-DPM,...,USG%,REB/100,AST/100,AST%,BLK/100,BLK%,STL/100,STL%,TOV/100,minutes
0,203999,Denver Nuggets,Nikola Jokic,Vet,6.71,0.03,5.30,1.41,4.36,4.35,...,0.311,15.895,14.200,0.456,1.018,0.018,2.192,0.021,5.383,1237.0
1,202695,Los Angeles Clippers,Kawhi Leonard,Vet,6.20,1.27,4.50,1.70,6.89,5.16,...,0.329,9.477,5.152,0.191,0.908,0.018,2.661,0.026,3.308,1211.0
2,1628983,Oklahoma City Thunder,Shai Gilgeous-Alexander,Vet,6.06,-0.19,5.47,0.59,3.64,4.14,...,0.323,6.408,8.223,0.302,1.164,0.021,2.078,0.021,3.327,1632.0
3,203507,Milwaukee Bucks,Giannis Antetokounmpo,Vet,6.01,-0.20,4.33,1.68,4.33,3.68,...,0.327,17.486,8.726,0.320,1.355,0.023,1.534,0.015,5.010,876.0
4,1628378,Cleveland Cavaliers,Donovan Mitchell,Vet,5.42,0.73,5.59,-0.17,3.39,4.73,...,0.314,6.653,8.057,0.272,0.319,0.010,2.107,0.021,4.028,1622.0


In [6]:
merged["minutes"] = merged["minutes"].fillna(0)
merged["minutes"].describe()

count     548.000000
mean      679.507299
std       518.318028
min         0.000000
25%       176.500000
50%       634.000000
75%      1093.500000
max      1866.000000
Name: minutes, dtype: float64

In [7]:
out_path = os.path.join(PROC_DIR, "darko_with_minutes_2025_26.csv")
merged.to_csv(out_path, index=False)

print("Saved:", out_path)

Saved: ../data/processed/darko_with_minutes_2025_26.csv


In [8]:
merged[["player_name","team","impact_dpm","minutes"]].sample(12)

,player_name,team,impact_dpm,minutes
460,RayJ Dennis,Atlanta Hawks,-2.14,177.0
111,Onyeka Okongwu,Atlanta Hawks,0.62,1503.0
104,Josh Okogie,Houston Rockets,0.70,979.0
48,Deni Avdija,Portland Trail Blazers,1.70,1523.0
46,Jalen Suggs,Orlando Magic,1.80,755.0
481,Jett Howard,Orlando Magic,-2.31,324.0
215,Justin Champagnie,Washington Wizards,-0.35,878.0
299,Noah Penda,Orlando Magic,-1.05,476.0
453,PJ Hall,Charlotte Hornets,-2.08,154.0
199,Santi Aldama,Memphis Grizzlies,-0.27,1198.0
